# Kaggle Notebook: Global LightGBM Baseline

This notebook trains a single global LightGBM model on all training wells and generates a Kaggle submission file.

In [ ]:
import os
from glob import glob
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except ImportError:
    !pip -q install lightgbm
    import lightgbm as lgb

SEED = 42
TRAIN_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/train'
TEST_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/test'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv'
RUN_EVALUATION = True
TRAIN_WELLS = 500
EVAL_WELLS = 100
CV_FOLDS = 3
HOLDOUT_WELLS = 60


In [ ]:
def safe_normalize(values):
    values = np.asarray(values, dtype=float)
    return (values - values.mean()) / (values.std() + 1e-8)


def infer_typewell_alignment(hw, tw):
    gr = hw['GR'].fillna(100).values.astype(float)
    tw_gr = tw['GR'].fillna(100).values.astype(float)
    tw_tvt = tw['TVT'].values.astype(float)

    gr_diff = np.abs(tw_gr.reshape(-1, 1) - gr.reshape(1, -1))
    nearest_idx = np.argmin(gr_diff, axis=0)
    matched_tvt = tw_tvt[nearest_idx]
    local_residual = gr - tw_gr[nearest_idx]

    return matched_tvt, nearest_idx.astype(float), local_residual


def compute_anchor_features(hw, fallback_tvt):
    n = len(hw)
    known_mask = hw['TVT_input'].notna().values
    anchor_features = {
        'dist_to_known': np.full(n, n, dtype=float),
        'nearest_known_tvt': np.full(n, fallback_tvt, dtype=float),
        'known_mask': known_mask.astype(int),
        'left_known_dist': np.full(n, n, dtype=float),
        'right_known_dist': np.full(n, n, dtype=float),
        'left_known_tvt': np.full(n, fallback_tvt, dtype=float),
        'right_known_tvt': np.full(n, fallback_tvt, dtype=float),
        'anchor_span': np.full(n, n, dtype=float),
        'anchor_position': np.full(n, 0.5, dtype=float),
        'interp_known_tvt': np.full(n, fallback_tvt, dtype=float),
    }

    if not known_mask.any():
        return anchor_features

    known_idx = np.where(known_mask)[0]
    known_tvt = hw.loc[known_mask, 'TVT_input'].values.astype(float)
    positions = np.arange(n)

    nearest_slot = np.argmin(np.abs(positions.reshape(-1, 1) - known_idx.reshape(1, -1)), axis=1)
    anchor_features['dist_to_known'] = np.abs(positions - known_idx[nearest_slot]).astype(float)
    anchor_features['nearest_known_tvt'] = known_tvt[nearest_slot]

    left_slot = np.searchsorted(known_idx, positions, side='right') - 1
    right_slot = np.searchsorted(known_idx, positions, side='left')

    left_exists = left_slot >= 0
    right_exists = right_slot < len(known_idx)

    if left_exists.any():
        left_idx = known_idx[left_slot[left_exists]]
        left_tvt = known_tvt[left_slot[left_exists]]
        anchor_features['left_known_dist'][left_exists] = positions[left_exists] - left_idx
        anchor_features['left_known_tvt'][left_exists] = left_tvt

    if right_exists.any():
        right_idx = known_idx[right_slot[right_exists]]
        right_tvt = known_tvt[right_slot[right_exists]]
        anchor_features['right_known_dist'][right_exists] = right_idx - positions[right_exists]
        anchor_features['right_known_tvt'][right_exists] = right_tvt

    between_mask = left_exists & right_exists
    if between_mask.any():
        left_idx = known_idx[left_slot[between_mask]]
        right_idx = known_idx[right_slot[between_mask]]
        left_tvt = known_tvt[left_slot[between_mask]]
        right_tvt = known_tvt[right_slot[between_mask]]
        span = np.maximum(right_idx - left_idx, 1)
        position = (positions[between_mask] - left_idx) / span
        anchor_features['anchor_span'][between_mask] = span.astype(float)
        anchor_features['anchor_position'][between_mask] = position
        anchor_features['interp_known_tvt'][between_mask] = left_tvt + position * (right_tvt - left_tvt)

    left_only_mask = left_exists & ~right_exists
    if left_only_mask.any():
        anchor_features['interp_known_tvt'][left_only_mask] = anchor_features['left_known_tvt'][left_only_mask]

    right_only_mask = right_exists & ~left_exists
    if right_only_mask.any():
        anchor_features['interp_known_tvt'][right_only_mask] = anchor_features['right_known_tvt'][right_only_mask]

    anchor_features['left_known_dist'] = anchor_features['left_known_dist'].astype(float)
    anchor_features['right_known_dist'] = anchor_features['right_known_dist'].astype(float)
    anchor_features['anchor_span'] = anchor_features['anchor_span'].astype(float)
    anchor_features['anchor_position'] = anchor_features['anchor_position'].astype(float)

    return anchor_features


def build_global_features(hw, tw, well_id=None):
    features = {}

    z = hw['Z'].values.astype(float)
    md = hw['MD'].values.astype(float)
    x = hw['X'].values.astype(float)
    y = hw['Y'].values.astype(float)
    features['Z'] = z
    features['MD'] = md
    features['X'] = x
    features['Y'] = y
    features['XY_radius'] = np.sqrt((x - x.mean())**2 + (y - y.mean())**2)

    gr = hw['GR'].fillna(100).values.astype(float)
    features['GR'] = gr
    features['GR_norm'] = safe_normalize(gr)

    gr_series = pd.Series(gr)
    for w in [5, 10, 20, 50]:
        roll_mean = gr_series.rolling(w, center=True, min_periods=1).mean().values
        roll_std = gr_series.rolling(w, center=True, min_periods=1).std().fillna(0).values
        features[f'GR_roll_mean_{w}'] = roll_mean
        features[f'GR_roll_std_{w}'] = roll_std
        features[f'GR_delta_roll_mean_{w}'] = gr - roll_mean

    gr_grad = np.gradient(gr)
    features['GR_grad'] = gr_grad
    features['GR_grad_abs'] = np.abs(gr_grad)
    features['GR_curvature'] = np.gradient(gr_grad)

    features['Z_norm'] = (z - z.min()) / (z.max() - z.min() + 1e-8)
    features['MD_norm'] = (md - md.min()) / (md.max() - md.min() + 1e-8)
    features['Z_centered'] = z - z.mean()
    features['MD_centered'] = md - md.mean()
    features['Z_MD_ratio'] = z / (md + 1e-8)

    tw_tvt = tw['TVT'].values.astype(float)
    tw_gr = tw['GR'].fillna(100).values.astype(float)
    features['tw_tvt_range'] = tw_tvt.max() - tw_tvt.min()
    features['tw_tvt_min'] = tw_tvt.min()
    features['tw_tvt_max'] = tw_tvt.max()
    features['tw_gr_std'] = tw_gr.std()
    features['tw_gr_mean'] = tw_gr.mean()

    matched_tvt, matched_idx, local_residual = infer_typewell_alignment(hw, tw)
    features['tw_tvt_from_gr'] = matched_tvt
    features['tw_index_from_gr'] = matched_idx
    features['tw_index_norm'] = matched_idx / max(len(tw_tvt) - 1, 1)
    features['gr_match_residual'] = local_residual
    features['tw_tvt_residual_to_min'] = matched_tvt - tw_tvt.min()

    anchor_features = compute_anchor_features(hw, fallback_tvt=tw_tvt.mean())
    features.update(anchor_features)
    features['interp_minus_nearest_anchor'] = features['interp_known_tvt'] - features['nearest_known_tvt']
    features['tw_tvt_minus_interp_anchor'] = matched_tvt - features['interp_known_tvt']
    features['tw_tvt_minus_nearest_anchor'] = matched_tvt - features['nearest_known_tvt']

    return pd.DataFrame(features).replace([np.inf, -np.inf], np.nan).fillna(0.0)


In [ ]:
def train_global_model(train_dir, max_wells=500):
    hw_files = glob(os.path.join(train_dir, '*__horizontal_well.csv'))

    if max_wells and len(hw_files) > max_wells:
        import random
        random.seed(SEED)
        hw_files = random.sample(hw_files, max_wells)

    all_X = []
    all_y = []
    all_groups = []
    well_sizes = {}

    print(f'Loading {len(hw_files)} wells for global training...')

    for i, f in enumerate(hw_files):
        well_id = os.path.basename(f).replace('__horizontal_well.csv', '')
        tw_path = os.path.join(train_dir, f'{well_id}__typewell.csv')
        if not os.path.exists(tw_path):
            continue

        hw = pd.read_csv(f)
        tw = pd.read_csv(tw_path)
        if 'TVT' not in hw.columns:
            continue

        features = build_global_features(hw, tw, well_id)
        all_X.append(features.values)
        all_y.append(hw['TVT'].values.astype(float))
        all_groups.extend([well_id] * len(hw))
        well_sizes[well_id] = len(hw)

        if (i + 1) % 100 == 0:
            print(f'  Loaded {i + 1}/{len(hw_files)} wells...')

    X = np.vstack(all_X)
    y = np.concatenate(all_y)
    groups = np.array(all_groups)
    feature_names = features.columns.tolist()

    print(f'Training data: X={X.shape}, y={y.shape}, {len(np.unique(groups))} wells')

    lgb_params = {
        'objective': 'regression',
        'metric': 'l2',
        'boosting_type': 'gbdt',
        'num_leaves': 127,
        'learning_rate': 0.03,
        'feature_fraction': 0.85,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 120,
        'lambda_l1': 0.5,
        'lambda_l2': 2.0,
        'max_bin': 255,
        'verbose': -1,
        'random_state': SEED,
        'n_jobs': -1,
    }

    unique_groups = np.unique(groups)
    holdout_count = min(HOLDOUT_WELLS, max(1, len(unique_groups) // 8))
    gss = GroupShuffleSplit(n_splits=1, test_size=holdout_count, random_state=SEED)
    holdout_group_idx = next(gss.split(unique_groups, groups=unique_groups))[1]
    holdout_groups = set(unique_groups[holdout_group_idx])
    holdout_mask = np.isin(groups, list(holdout_groups))
    train_mask = ~holdout_mask

    X_train = X[train_mask]
    y_train = y[train_mask]
    groups_train = groups[train_mask]
    X_holdout = X[holdout_mask]
    y_holdout = y[holdout_mask]

    group_weights = np.array([1.0 / np.sqrt(well_sizes[g]) for g in groups_train], dtype=float)
    group_weights *= len(group_weights) / group_weights.sum()

    kfold = GroupKFold(n_splits=CV_FOLDS)
    models = []
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train, groups_train)):
        print(f'Training fold {fold + 1}/{CV_FOLDS}...')
        train_data = lgb.Dataset(
            X_train[train_idx],
            label=y_train[train_idx],
            weight=group_weights[train_idx],
            feature_name=feature_names,
        )
        val_data = lgb.Dataset(
            X_train[val_idx],
            label=y_train[val_idx],
            feature_name=feature_names,
            reference=train_data,
        )

        model = lgb.train(
            lgb_params,
            train_data,
            num_boost_round=900,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(80), lgb.log_evaluation(100)],
        )
        models.append(model)

        fold_pred = model.predict(X_train[val_idx], num_iteration=model.best_iteration)
        fold_rmse = np.sqrt(np.mean((fold_pred - y_train[val_idx]) ** 2))
        fold_scores.append(fold_rmse)
        print(f'  Fold {fold + 1} RMSE: {fold_rmse:.3f}')

    holdout_preds = np.mean(
        [model.predict(X_holdout, num_iteration=model.best_iteration) for model in models],
        axis=0,
    )
    holdout_rmse = np.sqrt(np.mean((holdout_preds - y_holdout) ** 2))
    print(f'Holdout RMSE: {holdout_rmse:.3f}')
    print(f'CV RMSE mean={np.mean(fold_scores):.3f}, std={np.std(fold_scores):.3f}')

    return models, feature_names


def predict_well_global(well_id, hw, tw, models, feature_names):
    features = build_global_features(hw, tw, well_id)
    X = features.values
    preds = [model.predict(X, num_iteration=model.best_iteration) for model in models]
    pred = np.mean(preds, axis=0)
    return np.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)


def evaluate_method(predict_func, train_dir, method_name, n_wells=None, save_results=None, **kwargs):
    hw_files = glob(os.path.join(train_dir, '*__horizontal_well.csv'))

    if n_wells:
        import random
        random.seed(SEED)
        hw_files = random.sample(hw_files, min(n_wells, len(hw_files)))

    print(f'Evaluating {method_name} on {len(hw_files)} candidate wells...')

    rmses = []
    maes = []
    all_results = []

    for f in hw_files:
        well_id = os.path.basename(f).replace('__horizontal_well.csv', '')
        tw_path = os.path.join(train_dir, f'{well_id}__typewell.csv')
        if not os.path.exists(tw_path):
            continue

        hw = pd.read_csv(f)
        tw = pd.read_csv(tw_path)
        predict_mask = hw['TVT_input'].isna()
        if predict_mask.sum() < 1:
            continue

        tvt_pred = np.asarray(predict_func(well_id, hw, tw, **kwargs), dtype=float)
        y_true = hw.loc[predict_mask, 'TVT'].values.astype(float)
        y_pred = tvt_pred[predict_mask]

        if np.isnan(y_pred).any() or np.isinf(y_pred).any():
            continue

        rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
        mae = np.mean(np.abs(y_pred - y_true))
        rmses.append(rmse)
        maes.append(mae)

        for i, (true_val, pred_val) in enumerate(zip(y_true, y_pred)):
            row_idx = hw.index[predict_mask][i]
            all_results.append({
                'well_id': well_id,
                'point_idx': row_idx,
                'MD': hw.loc[row_idx, 'MD'],
                'Z': hw.loc[row_idx, 'Z'],
                'GR': hw.loc[row_idx, 'GR'],
                'TVT_true': true_val,
                'TVT_pred': pred_val,
                'error': pred_val - true_val,
                'abs_error': abs(pred_val - true_val),
            })

    if rmses:
        print(f'\n=== {method_name} on {len(rmses)} wells ===')
        print(f'  RMSE: mean={np.mean(rmses):.3f}, median={np.median(rmses):.3f}, std={np.std(rmses):.3f}')
        print(f'  MAE:  mean={np.mean(maes):.3f}, median={np.median(maes):.3f}, std={np.std(maes):.3f}')
        print(f'  P10={np.percentile(rmses, 10):.1f}, P50={np.percentile(rmses, 50):.1f}, P90={np.percentile(rmses, 90):.1f}')

    if save_results and all_results:
        pd.DataFrame(all_results).to_csv(save_results, index=False)
        print(f'Results saved to: {save_results}')

    return rmses, maes


def generate_submission(test_dir, sample_sub_path, output_path, predict_func, **kwargs):
    sub = pd.read_csv(sample_sub_path)
    test_files = glob(os.path.join(test_dir, '*__horizontal_well.csv'))
    predictions = {}

    for f in test_files:
        well_id = os.path.basename(f).replace('__horizontal_well.csv', '')
        hw = pd.read_csv(f)
        tw = pd.read_csv(os.path.join(test_dir, f'{well_id}__typewell.csv'))
        tvt_pred = np.asarray(predict_func(well_id, hw, tw, **kwargs), dtype=float)
        predictions[well_id] = np.nan_to_num(tvt_pred, nan=0.0, posinf=0.0, neginf=0.0)

    for idx, row in sub.iterrows():
        well_id = row['id'].rsplit('_', 1)[0]
        row_idx = int(row['id'].rsplit('_', 1)[1])
        if well_id in predictions:
            sub.at[idx, 'tvt'] = predictions[well_id][row_idx]

    sub.to_csv(output_path, index=False)
    print(f'Submission saved to {output_path}')
    return sub


In [ ]:
models, feature_names = train_global_model(TRAIN_DIR, max_wells=TRAIN_WELLS)

if RUN_EVALUATION:
    evaluate_method(
        predict_well_global,
        TRAIN_DIR,
        'Global LGB',
        n_wells=EVAL_WELLS,
        models=models,
        feature_names=feature_names,
        save_results=None,
    )

submission = generate_submission(
    TEST_DIR,
    SAMPLE_SUB_PATH,
    '/kaggle/working/submission.csv',
    predict_well_global,
    models=models,
    feature_names=feature_names,
)

submission.head()
